PREÁMBULO TALLER 5: IMPORTACIONES Y GESTIÓN DE RUTAS.

In [ ]:
'''
Relacionar google drive con Collab, para poder usar mi drive como almacenamiento de archivos,
y como lugar donde van a reposar los outputs que se generen en este script.
'''

from google.colab import drive
drive.mount('/content/t5drive')
# Ejecutar el código anterior y aceptar lo solicitado; permisos y accesos

In [ ]:
# Para instalar el paquete minisom, no se encuentra disponible en el entorno de ejecucion
!pip install minisom

In [7]:
'''
Importación de paquetes módulos y funciones. Cabe aclarar que no es necesario
importarlastodos al inicio, solo que es una práctica heredada de la codificación
tradicional que ayuda a ser mas eficiente la interpretación del código así como
su lectura.
'''

import os # Para interactuar con el sistema operativo (rutas de archivos, crear directorios)
import pandas as pd # Para manipulación y análisis de datos (DataFrames)
import scipy.io as sio # Necesitamos scipy para leer archivos .mat
import matplotlib.pyplot as plt # Necesitamos matplotlib para graficar
from sklearn.cluster import KMeans # Necesitamos sklearn para aplicar K-means
import numpy as np
from sklearn.preprocessing import StandardScaler
from minisom import MiniSom
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, precision_score, recall_score


In [8]:
'''
Definir las rutas para guardar las figuras, tablas y notebooks.
'''

# Definir las rutas base
base = '/content/t5drive/MyDrive/taller5'
figs = os.path.join(base, 'figuras')
codes = os.path.join(base, 'codes')
datos = os.path.join(base, 'datos')

# Crear las rutas sino existen
os.makedirs(figs, exist_ok=True)
os.makedirs(codes, exist_ok=True)
os.makedirs(datos, exist_ok=True)

PRIMER PUNTO TALLER 5

In [ ]:
'''
Subir un archivo desde la máquina local a Colab.
'''
from google.colab import files

uploaded = files.upload()
fn = next(iter(uploaded.keys()))
print('Usuario subió el archivo "{name}" con tamaño {length} bytes'.format(
    name=fn, length=len(uploaded[fn])))

# Construir la ruta completa donde se guardará el archivo
ruta_archivo_subido = os.path.join(datos, fn)

# Guardar el archivo subido en el directorio 'datos'
with open(ruta_archivo_subido, 'wb') as f:
  f.write(uploaded[fn])

print(f"Archivo guardado en {ruta_archivo_subido}")


In [ ]:
'''
Cargar el archivo .mat, transformarlo a un DataFrame de pandas y guardarlo como .csv.
'''
ruta_archivo_subido = os.path.join(datos, 'data_clusters.mat')
try:
    mat_contents = sio.loadmat(ruta_archivo_subido)
    print(f"Archivo .mat '{os.path.basename(ruta_archivo_subido)}' cargado exitosamente.")

    '''
    Identificar los datos relevantes en el archivo .mat.
    Los archivos .mat pueden tener varias variables; necesitamos encontrar la que contiene los datos.
    Podemos inspeccionar las claves del diccionario cargado.
    '''
    # Asumimos que los datos están en una clave específica o en la primera clave que no sea metadatos
    data_key = None
    for key in mat_contents:
        if not key.startswith('__'): # Ignorar claves de metadatos
            data_key = key
            break

    if data_key:
        data = mat_contents[data_key]
        print(f"Datos encontrados bajo la clave: '{data_key}'")

        '''
        Convertir los datos a un DataFrame de pandas.
        Asumimos que los datos tienen dos columnas.
        '''
        # Verificar las dimensiones de los datos
        if data.shape[1] >= 2:
            df = pd.DataFrame(data[:, :2], columns=['columnaA', 'columnaB'])
            print("Datos convertidos a DataFrame de pandas con encabezados.")

            '''
            Definir la ruta para el archivo .csv de salida.
            Se guardará en el mismo directorio 'datos' con el mismo nombre base pero extensión .csv.
            '''
            nombre_base = os.path.splitext(os.path.basename(ruta_archivo_subido))[0]
            ruta_csv_salida = os.path.join(datos, f"{nombre_base}.csv")

            '''
            Guardar el DataFrame en formato .csv.
            '''
            df.to_csv(ruta_csv_salida, index=False)
            print(f"Datos guardados exitosamente en formato .csv en: {ruta_csv_salida}")
        else:
            print(f"Error: Los datos en el archivo .mat no tienen al menos dos columnas. Dimensiones encontradas: {data.shape}")

    else:
        print("Error: No se encontraron datos relevantes en el archivo .mat (claves sin metadatos).")

except FileNotFoundError:
    print(f"Error: El archivo '{ruta_archivo_subido}' no fue encontrado. Asegúrate de haber subido el archivo correctamente.")
except Exception as e:
    print(f"Ocurrió un error al procesar el archivo .mat: {e}")

In [ ]:
'''
Visualizar los datos en un gráfico de dispersión (scatter plot).
'''

'''
Crear el gráfico de dispersión.
Usamos 'columnaA' para el eje x y 'columnaB' para el eje y.
'''
plt.figure(figsize=(10, 6)) # Define el tamaño de la figura
plt.scatter(df['columnaA'], df['columnaB'], alpha=0.7) # Crea el scatter plot con transparencia

'''
Añadir etiquetas.
'''
plt.xlabel('Columna A')
plt.ylabel('Columna B')

'''
Guardar la figura en la ruta especificada.
'''
nombre_figura = 'scatter_plot.png' # Define un nombre para el archivo de la figura
ruta_figura = os.path.join(figs, nombre_figura) # Construye la ruta completa para guardar la figura
plt.savefig(ruta_figura) # Guarda la figura en formato PNG

print(f"Gráfico guardado en: {ruta_figura}")

'''
Mostrar el gráfico.
'''
plt.grid(True) # Añade una cuadrícula para mejor visualización
plt.show()

In [ ]:
'''
K - MEANS.
Generar un modelo de K-means con las siguientes características:
1. Probar en el rango de k(número de clusters) = 2 a 10.
2. Los centroides iniciales de los clusters se determinan con el
   algoritmo k-means++.
3. Los centroides se reiniciaran 50 veces, es decir, 50 corridas. n_init = 50
4. A partir de la semilla plantada que elige los centroides iniciales de cada
   iteración, ajecutar el algoritmo de LLoyd (asignar datos a un centroide y
   recalcular la media de las distancias en ese cluster) 300 veces. max_iter = 300.
   Poner adicionalmente un tope de iteración cuando ya la función objetivo (
   inercia) ya no se ajuste mas de 0.0001 con respecto a la iteración anterior.
   tol = 1e-4.
5. Capturar para el número de cluster que se está probando, la inercia al final
   de cada corrida (es decir, al final de un ciclo de iteracion), con la intención
   de tener la 'mejor' inercia para cada k probado. Y poder gráficar número de k
   vs inercia (Gráfico del codo).
'''

In [ ]:
# Lista para almacenar los valores de inercia para cada k
inertia_values = []

# Rango de k a probar (de 2 a 10)
k_range = range(2, 11)

# Iterar sobre el rango de k
for k in k_range:
    # Inicializar el modelo KMeans
    kmeans = KMeans(n_clusters=k,
                    init='k-means++',  # Usar k-means++ para la inicialización
                    n_init=50,         # Número de veces que se ejecutará k-means con diferentes centroides iniciales
                    max_iter=300,      # Máximo número de iteraciones para el algoritmo de Lloyd
                    tol=1e-4,          # Tolerancia para la convergencia
                    random_state=42)   # Semilla para reproducibilidad

    # Entrenar el modelo con los datos (asumimos que los datos están en el DataFrame df)
    # Asegúrate de que 'df' contiene las columnas 'columnaA' y 'columnaB'
    kmeans.fit(df[['columnaA', 'columnaB']])

    # Capturar la inercia (suma de distancias cuadradas de las muestras a su centro de clúster más cercano)
    inertia_values.append(kmeans.inertia_)

# Imprimir los valores de inercia para cada k
for k, inertia in zip(k_range, inertia_values):
    print(f'Inercia para k={k}: {inertia:.4f}')

# Graficar el método del codo
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_values, marker='o')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inercia')
plt.grid(True)

# Guardar la figura en la ruta especificada
nombre_figura_codo = 'elbow_method.png'
ruta_figura_codo = os.path.join(figs, nombre_figura_codo) # Usar la variable 'figs'
plt.savefig(ruta_figura_codo)

print(f"Gráfico del codo guardado en: {ruta_figura_codo}")

# Mostrar el gráfico
plt.show()

In [ ]:
'''
Dado que se tuvo que con k = 6 aumentar el número de clusters ya no resulta tan
provechoso, se determina reentrenar el modelo pero solo con k = 5.
'''

# Reentrenar el modelo KMeans con 6 clusters
kmeans_final = KMeans(n_clusters=6,
                      init='k-means++',
                      n_init=50,
                      max_iter=300,
                      tol=1e-4,
                      random_state=42)

# Predecir los clusters para cada punto de datos
df['cluster'] = kmeans_final.fit_predict(df[['columnaA', 'columnaB']])

# Obtener los centroides de los clusters
centroids = kmeans_final.cluster_centers_

# Visualizar los datos con los clusters coloreados
plt.figure(figsize=(10, 6))

# Graficar los puntos de datos coloreados por cluster
plt.scatter(df['columnaA'], df['columnaB'], c=df['cluster'], cmap='viridis', alpha=0.7)

# Graficar los centroides de los clusters
plt.scatter(centroids[:, 0], centroids[:, 1], marker='X', s=200, c='red', label='Centroides')

plt.xlabel('Columna A')
plt.ylabel('Columna B')
plt.title('Clustering K-means con k=6')
plt.legend()
plt.grid(True)

# Guardar la figura en la ruta especificada
nombre_figura_clusters = 'kmeans_k6_clusters.png'
ruta_figura_clusters = os.path.join(figs, nombre_figura_clusters)
plt.savefig(ruta_figura_clusters)

print(f"Gráfico de clusters guardado en: {ruta_figura_clusters}")

# Mostrar el gráfico
plt.show()

In [ ]:
'''
Para el mismo dataset, generar un modelo de Mapas Autorganizados (SOM,
por sus siglas en inglés) con las siguientes características:
0. Estandarizar las variables
1. Número de épocas igual a 100, dividada en dos partes, una con Rough
training (vecindad grande) de 20 épocas, y otra con Fine-tuning
(vecindad pequeña).
2. Retícula de forma cuadrada de 64 neuronas, siguiendo la regla
de que el número de neuronas es aproximadamente 5 * raiz(n)
3. Forma de vecindad hexagonal.
4. Tasa de aprendizaje 'alfa' decreciente que inicie en 0.5 y pueda
llegar como máximo 0.01.
5. Función de vecindad gaussiana
6. Radio de vecindad inicial igual a 4, radio de vecindad final
igual a 1.
7. La correción de los pesos puede ser la forma clásica con la que
se corrigen en estos casos: wi​(t+1)=wi​(t)+α(t)hci​(t)(x(t)−wi​(t))
  hci = vecindad gaussiana
  wi = vector de pesos en la neurona
  α = tasa de aprendizaje
  x = vector de entrada
  t = número de dato que ingresa a la retícula
8. Realizar gráficos U-Matrix y coloreo de clusters para visualizar el entrenamiento.
9. Calcular QE para evlauar el rendimiento del modelo.
'''

In [ ]:
'''
Implementación de Mapas Autorganizados (SOM).
'''

# Estandarizar las variables
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df[['columnaA', 'columnaB']])

# Definir parámetros del SOM
n_neurons = 64  # Retícula de forma cuadrada de 64 neuronas
grid_size = int(np.sqrt(n_neurons)) # Asumiendo retícula cuadrada
num_epochs = 100
learning_rate_start = 0.5
learning_rate_end = 0.01

# Radio de vecindad
neighborhood_radius_start = 4
neighborhood_radius_end = 1

# Forma de vecindad hexagonal y función de vecindad gaussiana
# Inicializar el SOM
som = MiniSom(x=grid_size, y=grid_size, input_len=data_scaled.shape[1],
              sigma=neighborhood_radius_start, learning_rate=learning_rate_start,
              neighborhood_function='gaussian', topology='hexagonal', random_seed=42)

'''
# Inicializar los pesos aleatoriamente --> Podría haberse ingresado valores que
directamente estuvieran en el dataset
'''
som.random_weights_init(data_scaled)

# Entrenar el SOM para el número total de épocas.
print("Iniciando entrenamiento del SOM...")
som.train_random(data_scaled, num_iteration=num_epochs, verbose=True)
print("\nEntrenamiento del SOM finalizado.")

# Realizar gráfico U-Matrix para visualizar el entrenamiento.
plt.figure(figsize=(grid_size, grid_size))
plt.pcolor(som.distance_map().T, cmap='bone_r')  # U-matrix
plt.colorbar()
nombre_figura_umatrix = 'som_umatrix.png'
ruta_figura_umatrix = os.path.join(figs, nombre_figura_umatrix)
plt.savefig(ruta_figura_umatrix)
print(f"Gráfico U-Matrix guardado en: {ruta_figura_umatrix}")
plt.show()

# Coloreo de clusters basado en la salida del SOM
# Obtener la neurona ganadora para cada punto de dato
winning_neurons = np.array([som.winner(x) for x in data_scaled])

# Asignar un cluster a cada neurona ganadora
# Por simplicidad, se usa el índice de la neurona ganadora como proxy para el cluster
neuron_clusters = np.ravel_multi_index(winning_neurons.T, (grid_size, grid_size))

# Visualizar los puntos de datos coloreados por el cluster asignado basado en la neurona ganadora
plt.figure(figsize=(10, 10))
plt.scatter(data_scaled[:, 0], data_scaled[:, 1], c=neuron_clusters, cmap='viridis', s=50, alpha=0.7)
plt.xlabel('Columna A (Estandarizada)')
plt.ylabel('Columna B (Estandarizada)')
# plt.colorbar(label='ID neurona)')
plt.grid(True)

# Guardar la figura de coloreo de clusters
nombre_figura_som_clusters = 'som_cluster_coloring.png'
ruta_figura_som_clusters = os.path.join(figs, nombre_figura_som_clusters)
plt.savefig(ruta_figura_som_clusters)
print(f"Gráfico de Coloreo de Clusters del SOM guardado en: {ruta_figura_som_clusters}")
plt.show()

# Calcular QE para evlauar el rendimiento del modelo.
qe = som.quantization_error(data_scaled)
print(f"\nError de Cuantización (QE): {qe:.4f}")

SEGUNDO PUNTO TALLER 5

In [ ]:
'''
Subir un archivo desde la máquina local a Colab.
El código a continuación se debe ejecutar por cada archivo a subir. S podría
facilitar con un ciclo for, pero al ser solo dos, creo, es mas eficiente de
esta forma.
'''
from google.colab import files

uploaded = files.upload()
fn = next(iter(uploaded.keys()))
print('Usuario subió el archivo "{name}" con tamaño {length} bytes'.format(
    name=fn, length=len(uploaded[fn])))

# Construir la ruta completa donde se guardará el archivo
ruta_archivo_subido = os.path.join(datos, fn)

# Guardar el archivo subido en el directorio 'datos'
with open(ruta_archivo_subido, 'wb') as f:
  f.write(uploaded[fn])

print(f"Archivo guardado en {ruta_archivo_subido}")

In [ ]:
'''
Cargar los archivos .txt en DataFrames de pandas y asignar nombres de columnas
conforme se define en el archivo avila-description.txt
'''

# Definir los nombres de las columnas
column_names = [f'f{i}' for i in range(1, 11)] + ['clase']
ruta_avila_ts = os.path.join(datos, 'avila-ts.txt')
ruta_avila_tr = os.path.join(datos, 'avila-tr.txt')

try:
    # Cargar el archivo avila-ts.txt en un DataFrame
    df_ts = pd.read_csv(ruta_avila_ts, header=None, names=column_names)
    print(f"Archivo '{os.path.basename(ruta_avila_ts)}' cargado exitosamente.")

    # Cargar el archivo avila-tr.txt en un DataFrame
    df_tr = pd.read_csv(ruta_avila_tr, header=None, names=column_names)
    print(f"Archivo '{os.path.basename(ruta_avila_tr)}' cargado exitosamente.")

    # Mostrar el tamaño de cada DataFrame
    print(f"\nTamaño del DataFrame '{os.path.basename(ruta_avila_ts)}': {df_ts.shape}")
    print(f"Tamaño del DataFrame '{os.path.basename(ruta_avila_tr)}': {df_tr.shape}")

    # Mostrar los tipos de datos para verificar
    print("\nTipos de datos en df_ts:")
    print(df_ts.dtypes)
    print("\nTipos de datos en df_tr:")
    print(df_tr.dtypes)

except FileNotFoundError as e:
    print(f"Error: Uno de los archivos no fue encontrado. {e}")
except Exception as e:
    print(f"Ocurrió un error al cargar o procesar los archivos: {e}")

In [ ]:
'''
Crear nuevos DataFrames eliminando registros con clases las clases minoritarias
de acuerdo al archivo avila-description.txt.
'''

# Clases a eliminar
classes_to_remove = ['B', 'C', 'W', 'Y']

# Crear nuevos DataFrames excluyendo las clases especificadas
df_ts_filtered = df_ts[~df_ts['clase'].isin(classes_to_remove)].copy()
df_tr_filtered = df_tr[~df_tr['clase'].isin(classes_to_remove)].copy()

# Mostrar el tamaño de los DataFrames filtrados para verificar la eliminación
print(f"Tamaño del DataFrame 'df_ts_filtered' después de eliminar clases: {df_ts_filtered.shape}")
print(f"Tamaño del DataFrame 'df_tr_filtered' después de eliminar clases: {df_tr_filtered.shape}")

# Opcional: Mostrar las clases únicas restantes para verificar
print("\nClases únicas restantes en df_ts_filtered:")
print(df_ts_filtered['clase'].unique())
print("\nClases únicas restantes en df_tr_filtered:")
print(df_tr_filtered['clase'].unique())

In [ ]:
'''
Implementación de K-means en el dataset filtrado de entrenamiento. Utilizar los
siguientes parámetros:
1. Probar en el rango de k(número de clusters) = 2 a 25.
2. Los centroides iniciales de los clusters se determinan con el algoritmo k-means++.
3. Los centroides se reiniciaran con semillas distintas.
4. max_iter = 300.
5. tol = 1e-4.
6. alfa minimo de 0.01
'''

# Lista para almacenar los valores de inercia para cada k
inertia_values_tr = []

# Rango de k a probar (de 2 a 25)
k_range_tr = range(2, 26)

# Seleccionar solo las columnas de características para el clustering (f1 a f10)
X_tr_filtered = df_tr_filtered.drop('clase', axis=1)

# Iterar sobre el rango de k
for k in k_range_tr:
    # Inicializar el modelo KMeans
    kmeans_tr = KMeans(n_clusters=k,
                       init='k-means++',  # Usar k-means++ para la inicialización
                       n_init=30,         # Número de veces que se ejecutará k-means con diferentes centroides iniciales
                       max_iter=300,      # Máximo número de iteraciones para el algoritmo de Lloyd
                       tol=1e-4,          # Tolerancia para la convergencia
                       random_state=42)   # Semilla para reproducibilidad

    # Entrenar el modelo con los datos filtrados de entrenamiento
    kmeans_tr.fit(X_tr_filtered)

    # Capturar la inercia
    inertia_values_tr.append(kmeans_tr.inertia_)

# Imprimir los valores de inercia para cada k
for k, inertia in zip(k_range_tr, inertia_values_tr):
    print(f'Inercia para k={k}: {inertia:.4f}')

# Graficar el método del codo
plt.figure(figsize=(10, 6))
plt.plot(k_range_tr, inertia_values_tr, marker='o')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inercia')
plt.grid(True)

# Guardar la figura en la ruta especificada
nombre_figura_codo_tr = 'elbow_method_tr_filtered.png'
ruta_figura_codo_tr = os.path.join(figs, nombre_figura_codo_tr) # Usar la variable 'figs'
plt.savefig(ruta_figura_codo_tr)

print(f"Gráfico del codo guardado en: {ruta_figura_codo_tr}")

# Mostrar el gráfico
plt.show()

In [ ]:
'''
Reentrenar el modelo K-means con k = 15.
'''
from sklearn.cluster import KMeans
# import matplotlib.pyplot as plt # Removed plotting library import
# import os # Removed os import as it's not needed without saving file

# Reentrenar el modelo KMeans con 15 clusters
kmeans_final_tr = KMeans(n_clusters=15,
                         init='k-means++',
                         n_init=30,
                         max_iter=300,
                         tol=1e-4,
                         random_state=42)

# Entrenar el modelo con the original training features (before adding cluster column)
# Assuming X_tr_filtered still refers to the DataFrame without the 'cluster' column
kmeans_final_tr.fit(df_tr_filtered.drop('clase', axis=1))


print("Modelo K-means reentrenado con k=15.")


In [ ]:
'''
Crear un DataFrame con índice posicional, cluster asignado y clase original.
'''

# Obtener los clusters asignados por el modelo K-means final (k=15)
predicted_clusters_tr = kmeans_final_tr.predict(X_tr_filtered)

# Crear un nuevo DataFrame con el índice posicional, el cluster asignado y la clase original
# El índice posicional se puede derivar del índice del DataFrame filtrado
results_df_tr = pd.DataFrame({
    'indice_posicional': df_tr_filtered.index,
    'cluster_asignado': predicted_clusters_tr,
    'clase_original': df_tr_filtered['clase']
})

# Restablecer el índice de results_df_tr para obtener un índice posicional simple que comience desde 0
results_df_tr = results_df_tr.reset_index(drop=True)

print("DataFrame con índice posicional, cluster asignado y clase original creado.")
# Mostrar las primeras 5 filas del DataFrame sin el índice
print(results_df_tr.head().to_string(index=False))

In [ ]:
'''
Analizar la composición de cada cluster: contar datos por clase y encontrar la clase mayoritaria.
'''

# Agrupar por cluster y contar las ocurrencias de cada clase original
cluster_class_counts = results_df_tr.groupby('cluster_asignado')['clase_original'].value_counts().unstack(fill_value=0)

# Encontrar la clase mayoritaria para cada cluster
# idxmax(axis=1) encuentra la etiqueta de columna con el valor máximo para cada fila (cluster)
majority_class_per_cluster = cluster_class_counts.idxmax(axis=1)

# Añadir la clase mayoritaria como una nueva columna al DataFrame cluster_class_counts
cluster_class_counts['clase_mayoritaria'] = majority_class_per_cluster

print("Composición de cada cluster y clase mayoritaria:")
display(cluster_class_counts)


In [ ]:
'''
Etiquetar cada cluster con la clase mayoritaria y evaluar el rendimiento con una matriz de confusión.
'''

# Crear un diccionario mapeando el ID del cluster a la clase mayoritaria
cluster_to_label_mapping = majority_class_per_cluster.to_dict()

# Mapear la etiqueta de la clase mayoritaria a cada punto de dato en el DataFrame results_df_tr
results_df_tr['cluster_label'] = results_df_tr['cluster_asignado'].map(cluster_to_label_mapping)

print("Clusters etiquetados con su clase mayoritaria.")
# Mostrar las primeras filas del DataFrame actualizado para mostrar la nueva columna
display(results_df_tr.head())

# Generar la matriz de confusión
# Las etiquetas verdaderas son 'clase_original' y las etiquetas predichas son 'cluster_label'
# Asegúrate de que las clases estén en el mismo orden para ambos argumentos
true_labels = results_df_tr['clase_original']
predicted_labels = results_df_tr['cluster_label']

# Obtener todas las clases únicas en orden alfabético para asegurar consistencia
classes = sorted(true_labels.unique())

cm = confusion_matrix(true_labels, predicted_labels, labels=classes)

# Mostrar la matriz de confusión
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False) # Set colorbar to False
plt.xlabel('Etiqueta Predicha (Clase Mayoritaria del Cluster)')
plt.ylabel('Etiqueta Verdadera (Clase Original)')
plt.xticks(rotation=90)

# Guardar la figura en la ruta especificada
nombre_figura_mc = 'confusion_matrix_kmeans_k15.png'
ruta_figura_mc = os.path.join(figs, nombre_figura_mc)
plt.savefig(ruta_figura_mc)
print(f"Matriz de confusión guardada en: {ruta_figura_mc}")

plt.show()

In [ ]:
'''
Aplicar el modelo K-means entrenado al dataset de prueba filtrado y evaluar el rendimiento.
'''

# Seleccionar solo las columnas de características para la predicción (f1 a f10) en el dataset de prueba filtrado
X_ts_filtered = df_ts_filtered.drop('clase', axis=1)

# Predecir los clusters para cada punto de datos en el conjunto de prueba filtrado usando el modelo entrenado
predicted_clusters_ts = kmeans_final_tr.predict(X_ts_filtered)

# Crear un nuevo DataFrame con el índice posicional, el cluster asignado y la clase original para el dataset de prueba
results_df_ts = pd.DataFrame({
    'indice_posicional': df_ts_filtered.index,
    'cluster_asignado': predicted_clusters_ts,
    'clase_original': df_ts_filtered['clase']
})

# Restablecer el índice de results_df_ts para obtener un índice posicional simple que comience desde 0
results_df_ts = results_df_ts.reset_index(drop=True)

print("DataFrame con índice posicional, cluster asignado y clase original para el dataset de prueba creado.")
# Mostrar las primeras 5 filas del DataFrame sin el índice
print(results_df_ts.head().to_string(index=False))

# Analizar la composición de cada cluster en el dataset de prueba: contar datos por clase y encontrar la clase mayoritaria.
print("\nAnalizando la composición de los clusters en el dataset de prueba:")
# Agrupar por cluster y contar las ocurrencias de cada clase original en el dataset de prueba
cluster_class_counts_ts = results_df_ts.groupby('cluster_asignado')['clase_original'].value_counts().unstack(fill_value=0)

# Encontrar la clase mayoritaria para cada cluster en el dataset de prueba
majority_class_per_cluster_ts = cluster_class_counts_ts.idxmax(axis=1)

# Añadir la clase mayoritaria como una nueva columna al DataFrame cluster_class_counts_ts
cluster_class_counts_ts['clase_mayoritaria_ts'] = majority_class_per_cluster_ts

print("Composición de cada cluster y clase mayoritaria en el dataset de prueba:")
display(cluster_class_counts_ts)

# Etiquetar cada cluster con la clase mayoritaria encontrada en el dataset de prueba
# Crear un diccionario mapeando el ID del cluster a la clase mayoritaria del dataset de prueba
cluster_to_label_mapping_ts = majority_class_per_cluster_ts.to_dict()

# Mapear la etiqueta de la clase mayoritaria del dataset de prueba a cada punto de dato en el DataFrame results_df_ts
results_df_ts['cluster_label_ts'] = results_df_ts['cluster_asignado'].map(cluster_to_label_mapping_ts)

print("\nClusters en el dataset de prueba etiquetados con su clase mayoritaria (basada en datos de prueba).")
# Mostrar las primeras filas del DataFrame de prueba actualizado
display(results_df_ts.head())

# Generar la matriz de confusión para el dataset de prueba
# Las etiquetas verdaderas son 'clase_original' y las etiquetas predichas son 'cluster_label_ts'
true_labels_ts = results_df_ts['clase_original']
predicted_labels_ts = results_df_ts['cluster_label_ts']

# Obtener todas las clases únicas en orden alfabético para asegurar consistencia
classes_ts = sorted(true_labels_ts.unique())

cm_ts = confusion_matrix(true_labels_ts, predicted_labels_ts, labels=classes_ts)

# Mostrar la matriz de confusión para el dataset de prueba
disp_ts = ConfusionMatrixDisplay(confusion_matrix=cm_ts, display_labels=classes_ts)

fig_ts, ax_ts = plt.subplots(figsize=(10, 10))
disp_ts.plot(cmap=plt.cm.Blues, ax=ax_ts, colorbar=False) # Sin barra
plt.xlabel('Etiqueta Predicha (Clase Mayoritaria del Cluster - Prueba)')
plt.ylabel('Etiqueta Verdadera (Clase Original)')
plt.xticks(rotation=90)

# Guardar la figura en la ruta especificada
nombre_figura_mc_ts = 'confusion_matrix_kmeans_k15_test.png'
ruta_figura_mc_ts = os.path.join(figs, nombre_figura_mc_ts)
plt.savefig(ruta_figura_mc_ts)
print(f"Matriz de confusión para el dataset de prueba guardada en: {ruta_figura_mc_ts}")

plt.show()

In [ ]:
'''
Con el dataset df_tr_filtered ejecutar un modelo SOM con las siguientes características:
1. Épocas = 120
   - Rough/ordenamiento global: 20 épocas, vecindad amplia
   - Fine-tuning/local: 100 épocas, vecindad corta
2. Retícula rectangular.
3. El tamaño de la reticula será de entre 16 x 16 neuronas, hasta 36 x 36 neuronas,
variando de 1 en 1.
4. Tasa de aprendizaje:
  - inicio: 0.4
  - final: 0.2
5. Radio de vecindad depende del tamaño de la reticula que se este probando,
se va a calcular como la mitad del valor max(ancho, alto) de la re´ticula en
cuestión.
6. Función de vecindad: Gaussiana
7. Algoritmo de aprendizaje: batch SOM.
8. Realizar gráfico U-Matrix para visualizar el entrenamiento de cada retícula.
9. Calcular QE para evaluar el rendimiento de cada modelo/reticula. Construir
una gráfica que capture el QE por cada tamaño de retícula
'''


In [ ]:
# Seleccionar las columnas de características (f1 a f10) del DataFrame df_tr_filtered
X_train_scaled = df_tr_filtered.drop('clase', axis=1)
print("Datos para entrenamiento (sin estandarización):")
print("Forma de las características de entrenamiento:", X_train_scaled.shape)

In [12]:
# Definir el rango de tamaños de la retícula (de 16x16 a 36x36)
# Esto significa que el número de neuronas a lo largo de un lado de la retícula cuadrada irá de 16 a 36.
grid_sizes = range(16, 37)

# Inicializar listas vacías para almacenar los tamaños de la retícula y sus respectivos valores de QE
# Almacenaremos el número de neuronas a lo largo de un lado (grid_size)
som_grid_sizes = []
# Almacenaremos el número total de neuronas (grid_size * grid_size) para el eje x del gráfico
som_total_neurons = []
# Almacenaremos el error de cuantización para cada modelo SOM
som_qe_values = []

print(f"Rango de tamaños de retícula definido: de {grid_sizes[0]}x{grid_sizes[0]} a {grid_sizes[-1]}x{grid_sizes[-1]}.")
print(f"Listas vacías inicializadas para almacenar los resultados del entrenamiento del SOM.")

# Iniciar el bucle que itera a través de cada tamaño de retícula
# El bucle contendrá los pasos de entrenamiento y evaluación del SOM para cada tamaño de retícula

Rango de tamaños de retícula definido: de 16x16 a 36x36.
Listas vacías inicializadas para almacenar los resultados del entrenamiento del SOM.


In [ ]:
# Limpiar las listas para evitar acumulación de datos de ejecuciones anteriores
som_grid_sizes = []
som_total_neurons = []
som_qe_values = []

# Iterar a través de cada tamaño de retícula
for grid_size in grid_sizes:
    print(f"\nEntrenando SOM con retícula de {grid_size}x{grid_size}...")

    # Calcular el número total de neuronas
    n_neurons = grid_size * grid_size

    # Calcular el radio inicial de vecindad basado en el tamaño de la retícula
    neighborhood_radius_start = max(grid_size, grid_size) / 2
    neighborhood_radius_end = 1 # Como se especifica en las instrucciones

    # Definir el número de épocas para entrenamiento 'rough' y 'fine-tuning'
    total_epochs = 120
    rough_epochs = 20
    fine_tune_epochs = total_epochs - rough_epochs

    # Inicializar el SOM con parámetros iniciales para entrenamiento 'rough'
    # Pasar la dimensión de los datos de entrada (número de características) usando la forma del array de valores
    som = MiniSom(x=grid_size, y=grid_size, input_len=X_train_scaled.values.shape[1], # Use .values here
                  sigma=neighborhood_radius_start, learning_rate=0.4, # Tasa de aprendizaje inicial para entrenamiento 'rough'
                  neighborhood_function='gaussian', topology='hexagonal', random_seed=42)

    # Inicializar los pesos usando el array de NumPy
    som.random_weights_init(X_train_scaled.values) # Use .values here

    # Entrenar el SOM para la fase 'rough' usando el array de NumPy
    print("  Fase de entrenamiento 'Rough'...")
    som.train_batch(X_train_scaled.values, num_iteration=rough_epochs, verbose=False) # Use train_batch for rough training with .values

    # Ajustar la tasa de aprendizaje y sigma para la fase de 'fine-tuning'
    som._learning_rate = 0.2 # Tasa de aprendizaje para 'fine-tuning'
    som._sigma = neighborhood_radius_end # Sigma para 'fine-tuning'

    print("  Fase de 'Fine-tuning'...")
    # Continuar entrenando el SOM para la fase de 'fine-tuning' usando el array de NumPy
    som.train_batch(X_train_scaled.values, num_iteration=fine_tune_epochs, verbose=False) # Use train_batch for 'fine-tuning' with .values

    print("  Entrenamiento completado.")

    # Calcular el Error de Cuantización (QE) usando el SOM final (después de 'fine-tuning') y el array de NumPy
    qe = som.quantization_error(X_train_scaled.values) # Use X_train_scaled.values here
    print(f"  Error de Cuantización (QE) para la retícula {grid_size}x{grid_size}: {qe:.4f}")

    # Store the grid size (or total neurons) and QE value
    som_grid_sizes.append(grid_size)
    som_total_neurons.append(n_neurons)
    som_qe_values.append(qe)

    # Realizar gráfico U-Matrix para visualizar el entrenamiento del SOM fine-tuned.
    plt.figure(figsize=(grid_size, grid_size))
    plt.pcolor(som.distance_map().T, cmap='bone_r')  # U-matrix
    plt.colorbar()
    plt.title(f'U-Matrix for {grid_size}x{grid_size} SOM (Fine-tuned)')
    nombre_figura_umatrix = f'som_umatrix_{grid_size}x{grid_size}.png'
    ruta_figura_umatrix = os.path.join(figs, nombre_figura_umatrix)
    plt.savefig(ruta_figura_umatrix)
    plt.close() # Close the figure to prevent it from displaying in the output
    print(f"  Gráfico U-Matrix guardado en: {ruta_figura_umatrix}")

In [ ]:
# Crear el gráfico de QE vs. Tamaño de la Retícula (o número total de neuronas)
plt.figure(figsize=(12, 6))
plt.plot(som_total_neurons, som_qe_values, marker='o')
plt.xlabel('Número Total de Neuronas (Grid Size * Grid Size)')
plt.ylabel('Error de Cuantización (QE)')
plt.title('Error de Cuantización vs. Número Total de Neuronas del SOM')
plt.grid(True)

# Añadir etiquetas de tamaño de retícula al gráfico para mejor legibilidad en cada marcador
for i, txt in enumerate(som_grid_sizes):
    plt.annotate(f'{txt}x{txt}', (som_total_neurons[i], som_qe_values[i]), textcoords="offset points", xytext=(0,10), ha='center')


# Guardar el gráfico en el directorio de figuras especificado
nombre_figura_qe = 'som_qe_vs_neurons.png'
ruta_figura_qe = os.path.join(figs, nombre_figura_qe)
plt.savefig(ruta_figura_qe)

print(f"\nGráfico de QE vs. Número Total de Neuronas guardado en: {ruta_figura_qe}")

# Mostrar el gráfico
plt.show()

In [ ]:
'''
Entrenar un modelo SOM con retícula de 29x29 en el dataset df_tr_filtered.
'''

# Usar los datos ya estandarizados (asumiendo que X_train_scaled está disponible y estandarizado)
# Si no se ha hecho antes, estandarizar las características relevantes del dataset df_tr_filtered
# scaler = StandardScaler()
# X_train = df_tr_filtered.drop('clase', axis=1)
# X_train_scaled = scaler.fit_transform(X_train)
# Asumiendo que X_train_scaled está disponible de pasos anteriores

# Definir el tamaño específico de la retícula
grid_size = 29
n_neurons = grid_size * grid_size

# Definir otros parámetros del SOM como se usaron en la iteración
total_epochs = 120
rough_epochs = 20
fine_tune_epochs = total_epochs - rough_epochs
learning_rate_start = 0.4
learning_rate_end = 0.2
neighborhood_radius_start = max(grid_size, grid_size) / 2 # La mitad del tamaño de la retícula
neighborhood_radius_end = 1
# topology='hexagonal', neighborhood_function='gaussian', algorithm='batch' (por defecto en MiniSom train_batch)

# Inicializar el SOM con el tamaño de retícula de 28x28 y los parámetros
# Pasar la dimensión de los datos de entrada (número de características) usando la forma del array de valores
som_29x29 = MiniSom(x=grid_size, y=grid_size, input_len=X_train_scaled.values.shape[1], # Use .values here
                    sigma=neighborhood_radius_start, learning_rate=learning_rate_start,
                    neighborhood_function='gaussian', topology='hexagonal', random_seed=42)

# Inicializar los pesos usando el array de NumPy
som_29x29.random_weights_init(X_train_scaled.values) # Use .values here

# Entrenar el SOM en dos fases: entrenamiento 'rough' y 'fine-tuning' usando train_batch con el array de NumPy
print(f"Iniciando entrenamiento del SOM {grid_size}x{grid_size}...")

print("  Fase de entrenamiento 'Rough'...")
som_29x29.train_batch(X_train_scaled.values, num_iteration=rough_epochs, verbose=False) # Use .values here

# Ajustar la tasa de aprendizaje y sigma para la fase de 'fine-tuning'
som_29x29._learning_rate = learning_rate_end
som_29x29._sigma = neighborhood_radius_end

print("  Fase de 'Fine-tuning'...")
som_29x29.train_batch(X_train_scaled.values, num_iteration=fine_tune_epochs, verbose=False) # Use .values here

print("Entrenamiento del SOM finalizado.")

# Calcular el Error de Cuantización (QE) para este SOM específico usando el array de NumPy
qe_29x29 = som_29x29.quantization_error(X_train_scaled.values) # Use .values here
print(f"\nError de Cuantización (QE) para la retícula {grid_size}x{grid_size}: {qe_29x29:.4f}") # Use qe_28x28 here

# Opcionalmente, puedes generar el gráfico U-Matrix o el gráfico de Coloreo de Clusters para este SOM
# Realizar gráfico U-Matrix
plt.figure(figsize=(grid_size, grid_size))
plt.pcolor(som_29x29.distance_map().T, cmap='bone_r')
plt.colorbar()
plt.title(f'U-Matrix para SOM {grid_size}x{grid_size}')
nombre_figura_umatrix_29x29 = f'som_umatrix_{grid_size}x{grid_size}.png'
ruta_figura_umatrix_29x29 = os.path.join(figs, nombre_figura_umatrix_29x29)
plt.savefig(ruta_figura_umatrix_29x29)
plt.close()
print(f"Gráfico U-Matrix guardado en: {ruta_figura_umatrix_29x29}")

In [ ]:
'''
Cruzar BMU/neurona asignada por el SOM con la clase real y obtener una matriz "neurona vs. clase".
'''

# Obtener la neurona ganadora (BMU) para cada punto de dato en los datos de entrenamiento estandarizados
# som_29x29 es el modelo SOM entrenado del paso anterior
# X_train_scaled es el array de NumPy de datos de entrenamiento estandarizados
winning_neurons_coords = np.array([som_29x29.winner(x) for x in X_train_scaled.values]) # Usar .values para asegurar que es un array NumPy

# Convertir las coordenadas 2D de las neuronas ganadoras a un índice único para facilitar la agrupación
# Asumiendo un tamaño de retícula de 28x28 como se usó en el paso anterior
grid_size = 29
winning_neurons_flat_index = np.ravel_multi_index(winning_neurons_coords.T, (grid_size, grid_size))

# Obtener las etiquetas de clase originales del DataFrame de entrenamiento filtrado
original_classes_tr = df_tr_filtered['clase']

# Crear un DataFrame con el índice de la neurona ganadora y la clase original
bmu_class_df = pd.DataFrame({
    'winning_neuron_index': winning_neurons_flat_index,
    'original_class': original_classes_tr
})

# Crear una tabulación cruzada (matriz de contingencia) de neurona ganadora vs. clase original
neuron_class_counts = pd.crosstab(bmu_class_df['winning_neuron_index'], bmu_class_df['original_class'])

print("Matriz 'Neurona vs. Clase' con conteos:")
display(neuron_class_counts)

# Opcional: También puedes encontrar la clase mayoritaria para cada neurona a partir de esta matriz
# majority_class_per_neuron = neuron_class_counts.idxmax(axis=1)
# print("\nClase mayoritaria por neurona:")
# display(majority_class_per_neuron)

In [ ]:
'''
Asignar a cada neurona la clase mayoritaria y calcular su pureza.
'''

# Encontrar la clase mayoritaria para cada neurona
majority_class_per_neuron = neuron_class_counts.idxmax(axis=1)

# Calcular el número total de puntos de datos asignados a cada neurona
total_data_per_neuron = neuron_class_counts.sum(axis=1)

# Calcular el conteo de la clase mayoritaria para cada neurona
majority_class_counts = neuron_class_counts.max(axis=1)

# Calcular la pureza de cada neurona (conteo de la clase mayoritaria / total de datos en la neurona)
# Para evitar división por cero si alguna neurona no tiene datos asignados, podemos usar fill_value=0
neuron_purity = majority_class_counts / total_data_per_neuron.replace(0, np.nan).fillna(0) # Reemplazar 0 por NaN y luego NaN por 0 para manejar divisiones por cero

# Crear un DataFrame para almacenar los resultados
neuron_classification_results = pd.DataFrame({
    'clase_mayoritaria': majority_class_per_neuron,
    'pureza': neuron_purity,
    'total_datos_en_neurona': total_data_per_neuron
})

print("Clasificación de neuronas por clase mayoritaria y pureza:")
display(neuron_classification_results.head())

# Opcional: Mostrar neuronas con pureza alta
# print("\nNeuronas con pureza alta (> 0.8):")
# display(neuron_classification_results[neuron_classification_results['pureza'] > 0.8].head())

In [ ]:
'''
Generar una matriz de confusión para el dataset de entrenamiento usando las clases mayoritarias de las neuronas del SOM y calcular métricas de rendimiento.
'''
# Asumiendo que los DataFrames bmu_class_df y neuron_classification_results están disponibles de pasos anteriores

# Obtener la etiqueta de clase mayoritaria para cada neurona de neuron_classification_results
neuron_majority_class_mapping = neuron_classification_results['clase_mayoritaria'].to_dict()

# Mapear la etiqueta de clase mayoritaria de la neurona ganadora a cada punto de dato en bmu_class_df
bmu_class_df['predicted_class_som'] = bmu_class_df['winning_neuron_index'].map(neuron_majority_class_mapping)

# Obtener las etiquetas verdaderas (clases originales) y las etiquetas predichas (clase mayoritaria de la neurona ganadora)
true_labels_som_tr = bmu_class_df['original_class']
predicted_labels_som_tr = bmu_class_df['predicted_class_som']

# Obtener todas las clases únicas en orden alfabético para consistencia
classes_som_tr = sorted(true_labels_som_tr.unique())

# Generar la matriz de confusión
cm_som_tr = confusion_matrix(true_labels_som_tr, predicted_labels_som_tr, labels=classes_som_tr)

# Mostrar la matriz de confusión
disp_som_tr = ConfusionMatrixDisplay(confusion_matrix=cm_som_tr, display_labels=classes_som_tr)

fig_som_tr, ax_som_tr = plt.subplots(figsize=(10, 10))
disp_som_tr.plot(cmap=plt.cm.Blues, ax=ax_som_tr, colorbar=False) # No barra de color
plt.title('Matriz de Confusión del SOM (Dataset de Entrenamiento)')
plt.xlabel('Etiqueta Predicha (Clase Mayoritaria de la Neurona Ganadora)')
plt.ylabel('Etiqueta Verdadera (Clase Original)')
plt.xticks(rotation=90)

# Guardar la figura en la ruta especificada
nombre_figura_mc_som_tr = 'confusion_matrix_som_train.png'
ruta_figura_mc_som_tr = os.path.join(figs, nombre_figura_mc_som_tr)
plt.savefig(ruta_figura_mc_som_tr)
print(f"Matriz de confusión para el dataset de entrenamiento (SOM) guardada en: {ruta_figura_mc_som_tr}")

plt.show()

# Calcular e imprimir métricas de clasificación
print("\nMétricas de Clasificación (Dataset de Entrenamiento):")

# Accuracy
accuracy = accuracy_score(true_labels_som_tr, predicted_labels_som_tr)
print(f"Accuracy: {accuracy:.4f}")

# F1-score (usar average='weighted' para multiclase)
f1 = f1_score(true_labels_som_tr, predicted_labels_som_tr, average='weighted')
print(f"F1-Score (Ponderado): {f1:.4f}")

# Precision (usar average='weighted' para multiclase)
precision = precision_score(true_labels_som_tr, predicted_labels_som_tr, average='weighted')
print(f"Precision (Ponderado): {precision:.4f}")

# Recall (usar average='weighted' para multiclase)
recall = recall_score(true_labels_som_tr, predicted_labels_som_tr, average='weighted')
print(f"Recall (Ponderado): {recall:.4f}")

# Puedes añadir otras métricas si es necesario, por ejemplo, métricas por clase
# from sklearn.metrics import classification_report
# print("\nReporte de Clasificación:")
# print(classification_report(true_labels_som_tr, predicted_labels_som_tr, labels=classes_som_tr))

In [ ]:
'''
Evaluar el modelo SOM en el dataset de prueba filtrado y generar matriz de confusión.
'''

# Seleccionar solamente las caracterìsticas del dataset de prueba
X_ts_filtered_features = df_ts_filtered.drop('clase', axis=1)

# Agregar una revision para asegurar que la columna 'clsuter' no este presente antes de la tranformacion
if 'cluster' in X_ts_filtered_features.columns:
    X_ts_filtered_features = X_ts_filtered_features.drop('cluster', axis=1)
    print("Removed 'cluster' column from test features before scaling.")

# Usar los valores del DataFrame como array NumPy
X_ts_data = X_ts_filtered_features.values

# Obtener la neurona ganadora (BMU) para cada punto de dato en los datos de prueba
winning_neurons_coords_ts = np.array([som_29x29.winner(x) for x in X_ts_data]) # Usar X_ts_data aquí

# Convertir las coordenadas 2D de las neuronas ganadoras a un índice único
grid_size = 29
winning_neurons_flat_index_ts = np.ravel_multi_index(winning_neurons_coords_ts.T, (grid_size, grid_size))

# Mapear la etiqueta de clase mayoritaria de la neurona ganadora a cada punto de dato en el conjunto de prueba
# Use pd.Series for the index to ensure correct mapping with map()
predicted_labels_som_ts = pd.Series(winning_neurons_flat_index_ts).map(neuron_majority_class_mapping)

# # Manejar los valores NaN en las etiquetas predichas llenándolos con un marcador de posición
# predicted_labels_som_ts = predicted_labels_som_ts.fillna('Desconocido')

# Obtener las etiquetas verdaderas (clases originales) para los datos de prueba
true_labels_som_ts = df_ts_filtered['clase']

# Crear un DataFrame temporal para filtrar filas con etiquetas predichas NaN
temp_df = pd.DataFrame({'true': true_labels_som_ts, 'predicted': predicted_labels_som_ts})
temp_df_filtered = temp_df.dropna()

true_labels_som_ts_filtered = temp_df_filtered['true']
predicted_labels_som_ts_filtered = temp_df_filtered['predicted']

# --- Diagnóstico de tipos de datos y valores únicos ---
print("\nDiagnóstico antes de la Matriz de Confusión:")
print("Tipo de datos de true_labels_som_ts_filtered:", true_labels_som_ts_filtered.dtype)
print("Valores únicos en true_labels_som_ts_filtered:", true_labels_som_ts_filtered.unique())
print("Tipo de datos de predicted_labels_som_ts_filtered:", predicted_labels_som_ts_filtered.dtype)
print("Valores únicos en predicted_labels_som_ts_filtered:", predicted_labels_som_ts_filtered.unique())
# ----------------------------------------------------

# Obtener todas las clases únicas en orden alfabético para consistencia, incluyendo el marcador de posición
# Only use unique classes from the filtered data
classes_som_ts = sorted(true_labels_som_ts_filtered.unique())


# Generar la matriz de confusión para el dataset de prueba
cm_som_ts = confusion_matrix(true_labels_som_ts_filtered, predicted_labels_som_ts_filtered, labels=classes_som_ts)

# Mostrar la matriz de confusión para el dataset de prueba
disp_som_ts = ConfusionMatrixDisplay(confusion_matrix=cm_som_ts, display_labels=classes_som_ts)

fig_som_ts, ax_som_ts = plt.subplots(figsize=(10, 10))
disp_som_ts.plot(cmap=plt.cm.Blues, ax=ax_som_ts, colorbar=False) # Use ax_som_ts here, No color bar
plt.title('Matriz de Confusión del SOM (Dataset de Prueba)')
plt.xlabel('Etiqueta Predicha (Clase Mayoritaria de la Neurona Ganadora)')
plt.ylabel('Etiqueta Verdadera (Clase Original)')
plt.xticks(rotation=90)

# Guardar la figura en la ruta especificada
nombre_figura_mc_som_ts = 'confusion_matrix_som_test.png'
ruta_figura_mc_som_ts = os.path.join(figs, nombre_figura_mc_som_ts)
plt.savefig(ruta_figura_mc_som_ts)
print(f"Matriz de confusión para el dataset de prueba (SOM) guardada en: {ruta_figura_mc_som_ts}")
plt.show()